In [1]:
import os
import pickle as pkl

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from newutils.math import compute_corr_matrix
from oldutils.datasets import ROOT_FOLDER, HydroStaticFeaturesFiles
from config.Constants import ARTIFACTS

os.chdir(ROOT_FOLDER)

In [2]:
OUTDIR = ARTIFACTS / "Fast and Small"
OUTDIR.mkdir(parents=True, exist_ok=True)

# Explore the rivers

In [3]:
import contextily as ctx
print(list(ctx.providers))


['OpenStreetMap', 'MapTilesAPI', 'OpenSeaMap', 'OPNVKarte', 'OpenTopoMap', 'OpenRailwayMap', 'OpenFireMap', 'SafeCast', 'Stadia', 'Thunderforest', 'BaseMapDE', 'CyclOSM', 'Jawg', 'MapBox', 'MapTiler', 'TomTom', 'Esri', 'OpenWeatherMap', 'HERE', 'HEREv3', 'FreeMapSK', 'MtbMap', 'CartoDB', 'HikeBike', 'BasemapAT', 'nlmaps', 'NASAGIBS', 'NLS', 'JusticeMap', 'GeoportailFrance', 'OneMapSG', 'USGS', 'WaymarkedTrails', 'OpenAIP', 'OpenSnowMap', 'AzureMaps', 'SwissFederalGeoportal', 'TopPlusOpen', 'Gaode', 'Strava', 'OrdnanceSurvey', 'UN']


In [5]:
SUBSET_GAUGES =  [74425, 4005, 11571, 84215, 2115, 7015]

In [12]:
import geopandas as gpd
import polars as pl
import matplotlib.pyplot as plt
import cartopy.crs as ccrs
import cartopy.feature as cfeature

# 1. Load & filter
gdf = gpd.read_file("data/GTS_river_gauges.gpkg")
gdf["gauge_id"] = gdf["gauge_id"].astype(int)
ids = pl.read_csv("data/artifacts/merged_datasets/train_file_ids.csv")[
    "file_id"
].to_list()
gdf["lat"], gdf["lon"] = gdf.geometry.y, gdf.geometry.x
train = gdf[gdf["gauge_id"].isin(ids)]

# 2. Compute bounds
min_lon, min_lat, max_lon, max_lat = train.total_bounds
buffer = 0.1

# 3. Create large figure (dimensions in inches)
fig = plt.figure(figsize=(20, 18))
ax = plt.axes(projection=ccrs.PlateCarree())

# 4. Add high-res Natural Earth 10m features
ax.add_feature(cfeature.LAND.with_scale("10m"))
ax.add_feature(cfeature.COASTLINE.with_scale("50m"), linewidth=0.5)
ax.set_extent([min_lon - buffer, max_lon + buffer, min_lat - buffer, max_lat + buffer])

# 5. Plot gauge points (scaled for larger figure)
ax.scatter(
    train[train["gauge_id"].isin(SUBSET_GAUGES)]["lon"],
    train[train["gauge_id"].isin(SUBSET_GAUGES)]["lat"],
    s=100,
    c="red",
    edgecolor="black",
    linewidth=0.5,
    alpha=1,
    transform=ccrs.PlateCarree(),
    zorder=5,
    label="Gauges",
)

ax.scatter(
    train[~train["gauge_id"].isin(SUBSET_GAUGES)]["lon"],
    train[~train["gauge_id"].isin(SUBSET_GAUGES)]["lat"],
    s=100,
    c="blue",
    edgecolor="black",
    linewidth=0.5,
    alpha=0.5,
    transform=ccrs.PlateCarree(),
    zorder=2,
    label="Gauges",
)

ax.legend(loc="upper right")
ax.set_title("Training River Gauge Locations", pad=12)

# 6. **Save as PDF** (vector format)
plt.savefig(
    "train_gauges_cartopy_10m_large.pdf",  # output filename ends in .pdf
    format="pdf",  # explicitly PDF
    bbox_inches="tight",
)
plt.close()